# Use Case: Sentiment Analysis (NLP)

**User Prompt Example:**
```
Perform sentiment analysis on @[path/to/reviews.csv].
Classify reviews as positive or negative. Use VADER and BERT.
Export results and create visualizations.
Follow the data-science-lab skill.
```

**Goal Interpretation:**
| User Goal | Task Type | Recommended Models |
|-----------|-----------|-------------------|
| "sentiment analysis" | NLP classification | VADER, TextBlob, Transformer Models |

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Try VADER (will install if needed)
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'vaderSentiment'])
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

from visualize import plot_confusion_matrix

sns.set_style('whitegrid')

for d in ['results', 'images', 'data']:
    os.makedirs(d, exist_ok=True)

print('✅ Setup complete.')

---

## 2. Load & Explore Data

In [ ]:
# Generate sample review data (replace with actual data)
np.random.seed(42)
n_reviews = 1000

positive_texts = [
    "Great product! Loved it.", "Excellent quality, highly recommend!",
    "Amazing service, will buy again!", "Best purchase I've ever made.",
    "Very satisfied, exceeded expectations.", "Wonderful experience!",
    "Outstanding product, worth every penny.", "Fantastic! Five stars!",
    "Really impressed with this purchase.", "Could not be happier!"
]

negative_texts = [
    "Terrible product, waste of money.", "Very disappointed, do not buy.",
    "Awful experience, would not recommend.", "Worst purchase ever.",
    "Not satisfied, returned immediately.", "Horrible quality!",
    "Very bad, total disappointment.", "Avoid this product!",
    "Poor quality, not worth it.", "Regret buying this."
]

neutral_texts = [
    "It's okay, nothing special.", "Average product, as expected.",
    "Decent, does what it says.", "Not bad, not great.",
    "It works, but that's it.", "Fair for the price."
]

data = {
    'review_id': range(1, n_reviews + 1),
    'text': np.random.choice(positive_texts + negative_texts + neutral_texts, n_reviews),
}

# Add some noise/variation
for i in range(n_reviews):
    if np.random.random() < 0.3:
        data['text'][i] = data['text'][i] + " " + np.random.choice([
            "Very ", "Really ", "Absolutely ", "Not ", "Just ",
            "definitely", "probably", "maybe"
        ])

df = pd.DataFrame(data)

# Create labels (binary: positive/negative for simplicity)
df['label'] = df['text'].apply(lambda x: 'positive' if x in positive_texts else 
                               ('negative' if x in negative_texts else 'neutral'))

# Convert to binary for classification
df['binary_label'] = df['label'].apply(lambda x: 1 if x == 'positive' else 0)

print(f'Shape: {df.shape}')
print(f'\nLabel Distribution:')
print(df['label'].value_counts())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 4))
df['label'].value_counts().plot(kind='bar', color=['green', 'gray', 'red'])
plt.title('Review Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('images/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. Method 1: VADER Sentiment

In [ ]:
# VADER (Valence Aware Dictionary and sEntiment Reasoner)
# Best for social media/reviews text

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    scores = vader.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

def get_vader_score(text):
    return vader.polarity_scores(text)['compound']

df['vader_sentiment'] = df['text'].apply(get_vader_sentiment)
df['vader_score'] = df['text'].apply(get_vader_score)

print('=== VADER Results ===')
print(df[['text', 'label', 'vader_sentiment', 'vader_score']].head(10))

In [ ]:
# VADER accuracy (for positive/negative only)
df_binary = df[df['label'] != 'neutral'].copy()
vader_accuracy = (df_binary['vader_sentiment'] == df_binary['label']).mean()
print(f'VADER Accuracy (pos/neg only): {vader_accuracy:.2%}')

---

## 4. Method 2: TF-IDF + Logistic Regression

In [ ]:
# Text Vectorization using TF-IDF
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words='english')
X_tfidf = tfidf.fit_transform(df['text'])
y = df['binary_label']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size: {X_test.shape[0]}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print('=== Logistic Regression Results ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_lr):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
plot_confusion_matrix(cm, labels=['Negative', 'Positive'], 
                      title='Logistic Regression Confusion Matrix',
                      save_path='images/lr_confusion_matrix.png')
plt.show()

---

## 5. Method 3: Random Forest

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print('=== Random Forest Results ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_rf):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
plot_confusion_matrix(cm, labels=['Negative', 'Positive'], 
                      title='Random Forest Confusion Matrix',
                      save_path='images/rf_confusion_matrix.png')
plt.show()

---

## 6. Model Comparison

In [ ]:
# Compare all models
results = [
    {'model': 'VADER', 'accuracy': vader_accuracy, 'f1': vader_accuracy},
    {'model': 'Logistic Regression', 'accuracy': accuracy_score(y_test, y_pred_lr), 
     'f1': f1_score(y_test, y_pred_lr)},
    {'model': 'Random Forest', 'accuracy': accuracy_score(y_test, y_pred_rf), 
     'f1': f1_score(y_test, y_pred_rf)},
]

results_df = pd.DataFrame(results)
print('=== Model Comparison ===')
print(results_df)

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['accuracy'], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, results_df['f1'], width, label='F1 Score', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Sentiment Analysis Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results_df['model'])
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.savefig('images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 7. Feature Importance

In [ ]:
# Top features for Logistic Regression
feature_names = tfidf.get_feature_names_out()
coefficients = lr_model.coef_[0]

top_positive = np.argsort(coefficients)[-10:]
top_negative = np.argsort(coefficients)[:10]

print('Top Positive Words:')
for idx in reversed(top_positive):
    print(f'  {feature_names[idx]}: {coefficients[idx]:.4f}')

print('\nTop Negative Words:')
for idx in top_negative:
    print(f'  {feature_names[idx]}: {coefficients[idx]:.4f}')

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Positive features
pos_features = [feature_names[i] for i in top_positive]
pos_coeffs = [coefficients[i] for i in top_positive]
axes[0].barh(pos_features, pos_coeffs, color='green', alpha=0.7)
axes[0].set_title('Top Positive Features')
axes[0].set_xlabel('Coefficient')

# Negative features  
neg_features = [feature_names[i] for i in top_negative]
neg_coeffs = [abs(coefficients[i]) for i in top_negative]
axes[1].barh(neg_features, neg_coeffs, color='red', alpha=0.7)
axes[1].set_title('Top Negative Features')
axes[1].set_xlabel('|Coefficient|')

plt.tight_layout()
plt.savefig('images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Results Export

In [ ]:
# Export results
results_df.to_csv('results/sentiment_comparison.csv', index=False)

# Export full predictions
df['lr_prediction'] = lr_model.predict(X_tfidf)
df['rf_prediction'] = rf_model.predict(X_tfidf)
df.to_csv('data/reviews_with_sentiment.csv', index=False)

print('✅ Results saved:')
print('  - results/sentiment_comparison.csv')
print('  - data/reviews_with_sentiment.csv')

---

## 9. Conclusion

### Key Findings

**VADER:**
- Fast, no training required
- Good baseline for quick sentiment analysis
- Handles emojis and slang well

**Logistic Regression + TF-IDF:**
- Best balance of accuracy and interpretability
- Provides probability scores
- Fast inference

**Random Forest:**
- Handles non-linear relationships
- Good for complex datasets
- Less interpretable

### Recommendations
- Use VADER for quick prototyping
- Use Logistic Regression for production (fast + interpretable)
- Use Random Forest when accuracy is critical and interpretability is not